# Resultados y figuras para la presentación
### YOLO + SAHI sobre VisDrone — FLISOL 2026

Genera todos los **gráficos en alta resolución** que vas a pegar en las slides:
diagrama del pipeline, comparación de mAP, AP por tamaño, trade-off de slice y
comparación de postprocesamiento. Quedan en `outputs/figures/`.

Para el **mAP real** necesitas el modelo entrenado en VisDrone (`weights/visdrone_yolo11s.pt`).
Sin él, se generan igual las figuras que no dependen de mAP (pipeline, trade-off).

## 1. Setup

In [ ]:
!pip install -q ultralytics sahi pycocotools matplotlib
!git clone https://github.com/TU-USUARIO/yolo-sahi-flisol2026.git || echo 'ya clonado'
%cd yolo-sahi-flisol2026
import os, sys; sys.path.insert(0, 'src')

## 2. Datos + modelo

In [ ]:
!python scripts/download_visdrone.py
MODEL = 'weights/visdrone_yolo11s.pt' if os.path.exists('weights/visdrone_yolo11s.pt') else 'yolo11s.pt'
print('Modelo:', MODEL, '| mAP real requiere el modelo VisDrone')

## 3. Generar los CSV de resultados

Cada script guarda un CSV en `outputs/`. Ajusta `--limit` según el tiempo disponible.

In [ ]:
# Benchmark principal: mAP YOLO vs YOLO+SAHI (requiere modelo VisDrone)
!python src/evaluate.py --model $MODEL --limit 100

In [ ]:
# Trade-off de tamaño de slice (cualquier modelo)
!python src/slice_sweep.py --model $MODEL --limit 15 --sizes 320 512 640 768 1024

In [ ]:
# Comparación de postprocesamiento NMS / NMM / GREEDYNMM (mAP requiere modelo VisDrone)
!python src/compare_postprocess.py --model $MODEL --limit 60

## 4. Generar todas las figuras

In [ ]:
import importlib, figures; importlib.reload(figures)
figures.all_figures()

## 5. Previsualizar las figuras

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('outputs/figures/*.png')):
    print(f)
    display(Image(f))

## 6. Galería antes/después + descargar todo

In [ ]:
!python src/gallery.py --model $MODEL --limit 20 --top 6

# Empaquetar figuras + galería para bajar a tu compu y pegar en las slides
!zip -r resultados_slides.zip outputs/figures outputs/gallery outputs/*.csv
try:
    from google.colab import files; files.download('resultados_slides.zip')
except Exception as e:
    print('Descarga manual: resultados_slides.zip', e)